# CS605 Assignment: Role-playing Toxicity Detection

This notebook covers:
- **Step 1**: Exploratory Data Analysis (EDA)
- **Step 2**: Preprocessing and Feature Engineering

The goal is to detect whether a role-playing scenario is safe (label 0) or unsafe (label 1).

## 0. Install and import dependencies

In [1]:
!pip install pandas matplotlib seaborn scikit-learn wordcloud --quiet

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import warnings
from collections import Counter
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

SAFE_COLOR   = '#1D9E75'
UNSAFE_COLOR = '#D85A30'
NEUTRAL_COLOR = '#7F77DD'

print('All packages imported successfully.')

All packages imported successfully.


---
# Step 1: Exploratory Data Analysis

### 1.1 Load the dataset

Load `train.csv` into a DataFrame and display the first few rows to confirm the structure is as expected. The four columns are `id`, `movie`, `scenario`, and `label`.

In [4]:
#@title connect google drive folder

from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SMU_MITB_NLP/Assignment/

Mounted at /content/drive
/content/drive/MyDrive/SMU_MITB_NLP/Assignment


In [5]:
# mount Google Drive
DATA_PATH = 'train.csv'

df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(5)

Shape: (6410, 4)
Columns: ['id', 'movie', 'scenario', 'label']


,id,movie,scenario,label
0,0,The Shawshank Redemption,"User: ""What do you think about the guards at S...",1
1,1,The Shawshank Redemption,Scenario 0: Andy has a conversation with Red a...,0
2,2,The Shawshank Redemption,Scenario 1: Warden Norton belittles Andy for h...,1
3,3,The Shawshank Redemption,Scenario 2: Brooks expresses his concerns abou...,0
4,4,The Shawshank Redemption,"User: ""How do you feel about those who betraye...",1


### 1.2 Basic dataset statistics

Check data types, missing values, and the overall size of the dataset before doing anything else. Missing or null entries can silently corrupt tokenization later.

In [6]:
print('=== Data types ===')
print(df.dtypes)

print('\n=== Missing values ===')
print(df.isnull().sum())

print('\n=== Duplicate rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

print('\n=== Label value counts ===')
print(df['label'].value_counts())
print(f'\nLabel balance: {df["label"].value_counts(normalize=True).round(3).to_dict()}')

=== Data types ===
id           int64
movie       object
scenario    object
label        int64
dtype: object

=== Missing values ===
id          0
movie       0
scenario    1
label       0
dtype: int64

=== Duplicate rows ===
Duplicates: 0

=== Label value counts ===
label
0    3388
1    3022
Name: count, dtype: int64

Label balance: {0: 0.529, 1: 0.471}


* There is one row having missing scenario. Scenario cannot be imputed, therefore, we remove it.
* Label is quite balanced, but we need to check the sanity of these labels.

### 1.3 Label distribution

Visualise how safe and unsafe samples are distributed. A large imbalance would require strategies such as class weighting or oversampling, so it is important to check this before building any model.

In [ ]:
label_counts = df['label'].value_counts().sort_index()
label_names  = {0: 'Safe (0)', 1: 'Unsafe (1)'}
colors = [SAFE_COLOR, UNSAFE_COLOR]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
bars = axes[0].bar(
    [label_names[i] for i in label_counts.index],
    label_counts.values,
    color=colors, edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)
axes[0].set_title('Sample count by label')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, label_counts.max() * 1.15)
axes[0].spines[['top', 'right']].set_visible(False)

# Pie chart
axes[1].pie(
    label_counts.values,
    labels=[label_names[i] for i in label_counts.index],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Label proportion')

plt.tight_layout()
plt.suptitle('Label Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.show()

print('The dataset is nearly balanced (53% safe, 47% unsafe).')
print('Standard cross-entropy loss is appropriate — no class weighting required.')

### 1.4 Movie distribution

Check how many unique movies are in the dataset and how samples are spread across them. Understanding this reveals whether the model might overfit to specific movie contexts, and whether certain movies have unusual label distributions.

In [ ]:
print(f'Total unique movies: {df["movie"].nunique()}')
print(f'Samples per movie — mean: {df.groupby("movie").size().mean():.1f}, '
      f'min: {df.groupby("movie").size().min()}, '
      f'max: {df.groupby("movie").size().max()}')

movie_label = df.groupby('movie')['label'].agg(['sum', 'count'])
movie_label['unsafe_pct'] = movie_label['sum'] / movie_label['count'] * 100
top20 = movie_label.sort_values('count', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(top20))
width = 0.4

safe_counts   = top20['count'] - top20['sum']
unsafe_counts = top20['sum']

ax.bar(x, safe_counts,   width, label='Safe',   color=SAFE_COLOR,   edgecolor='white')
ax.bar(x, unsafe_counts, width, bottom=safe_counts, label='Unsafe', color=UNSAFE_COLOR, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=9)
ax.set_title('Top 20 movies by sample count (stacked by label)')
ax.set_ylabel('Sample count')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\nMovies with highest unsafe rate:')
print(movie_label.sort_values('unsafe_pct', ascending=False).head(5)[['count', 'unsafe_pct']].rename(
    columns={'count': 'total', 'unsafe_pct': 'unsafe_%'}).round(1))

print('\nMovies with lowest unsafe rate:')
print(movie_label.sort_values('unsafe_pct').head(5)[['count', 'unsafe_pct']].rename(
    columns={'count': 'total', 'unsafe_pct': 'unsafe_%'}).round(1))

### 1.5 Scenario text length analysis

Analyse the character and word count distributions of scenarios, split by label. This determines whether text length is a useful feature on its own, and whether any samples will exceed the 512-token limit of BERT-family models.

In [ ]:
df['char_len'] = df['scenario'].fillna('').str.len()
df['word_len'] = df['scenario'].fillna('').str.split().str.len()

print('=== Character length by label ===')
print(df.groupby('label')[['char_len']].describe().round(1))

print('\n=== Word length by label ===')
print(df.groupby('label')[['word_len']].describe().round(1))

# Transformer token budget check (rough: 1 word ≈ 1.3 tokens)
over_budget = (df['word_len'] > 350).sum()
print(f'\nSamples potentially over 512 tokens (>350 words): {over_budget}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, name in [(0, SAFE_COLOR, 'Safe'), (1, UNSAFE_COLOR, 'Unsafe')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['char_len'], bins=40, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[1].hist(subset['word_len'], bins=40, alpha=0.6, color=color, label=name, edgecolor='white')

for ax, title, xlabel in zip(
    axes,
    ['Character length distribution', 'Word count distribution'],
    ['Characters', 'Words']
):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 1.6 Scenario format types

The dataset contains scenarios written in three different formats. Knowing this is important because each format embeds information differently — direct dialogue (`User:` format) makes the harmful content explicit, while narrative descriptions (`Scenario N:` format) describe it in third person. The model must handle all three.

In [ ]:
def detect_format(text):
    text = str(text).strip()
    if text.startswith('User:') or text.startswith("User '") or text.startswith('User \"'):
        return 'User: dialogue'
    elif re.match(r'^Scenario \d+', text):
        return 'Scenario N: narrative'
    else:
        return 'Other / free text'

df['format_type'] = df['scenario'].apply(detect_format)

print('=== Format type counts ===')
print(df['format_type'].value_counts())

print('\n=== Label distribution within each format ===')
print(df.groupby('format_type')['label'].value_counts(normalize=True).round(3).unstack())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fmt_counts = df['format_type'].value_counts()
axes[0].barh(fmt_counts.index, fmt_counts.values, color=NEUTRAL_COLOR, edgecolor='white')
axes[0].set_title('Scenario format type counts')
axes[0].set_xlabel('Count')
axes[0].spines[['top', 'right']].set_visible(False)

fmt_label = df.groupby(['format_type', 'label']).size().unstack(fill_value=0)
fmt_label.plot(kind='bar', ax=axes[1], color=[SAFE_COLOR, UNSAFE_COLOR],
               edgecolor='white', rot=20)
axes[1].set_title('Safe vs Unsafe count by format type')
axes[1].set_ylabel('Count')
axes[1].legend(['Safe', 'Unsafe'])
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 1.7 Data quality check

Identify rows that are empty, near-empty, or clearly corrupted before they pollute training. Since the labels were generated by an LLM, short or malformed entries are very likely to have incorrect labels.

In [ ]:
empty_rows  = df[df['scenario'].fillna('').str.strip() == '']
short_rows  = df[(df['scenario'].fillna('').str.strip().str.len() > 0) &
                 (df['scenario'].fillna('').str.strip().str.len() < 20)]

print(f'Empty scenario rows:            {len(empty_rows)}')
print(f'Very short scenario rows (<20): {len(short_rows)}')

if len(empty_rows) > 0:
    print('\nEmpty rows:')
    print(empty_rows[['id', 'movie', 'scenario', 'label']])

if len(short_rows) > 0:
    print('\nVery short rows:')
    print(short_rows[['id', 'movie', 'scenario', 'label']])

print(f'\nTotal problematic rows to remove: {len(empty_rows) + len(short_rows)}')

### 1.8 Word frequency and signal analysis

Compare the most frequent words in safe versus unsafe scenarios. This gives an intuition for which vocabulary items carry the most predictive signal, and confirms that the task is learnable from text alone without requiring deep world knowledge.

In [ ]:
STOPWORDS = set((
    'the a an in is are was were you your i me my we our they them their '
    'it its he she his her and or but of to for with on at by from as if '
    'that this be have has had do does did not no will can could would '
    'should just like also what how who when where which user scenario '
    'says said tell told ask asked think thought know knew want wanted '
    'one two three four five about just more very really going get got '
    'there here than then so too much many any some all up down out off '
    'over after before because while though even still yet only both '
    'between through during against into onto upon within without '
    "don't doesn't can't won't isn't aren't wasn't weren't"
).split())

def get_top_words(texts, n=30):
    words = re.findall(r'\b[a-z]{4,}\b', ' '.join(texts).lower())
    return Counter(w for w in words if w not in STOPWORDS).most_common(n)

safe_texts   = df[df['label'] == 0]['scenario'].fillna('').tolist()
unsafe_texts = df[df['label'] == 1]['scenario'].fillna('').tolist()

safe_top   = get_top_words(safe_texts)
unsafe_top = get_top_words(unsafe_texts)

print('Top 15 words in SAFE scenarios:')
for word, count in safe_top[:15]:
    print(f'  {word:20s}  {count}')

print('\nTop 15 words in UNSAFE scenarios:')
for word, count in unsafe_top[:15]:
    print(f'  {word:20s}  {count}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, top_words, color, title in [
    (axes[0], safe_top[:20],   SAFE_COLOR,   'Top 20 words in safe scenarios'),
    (axes[1], unsafe_top[:20], UNSAFE_COLOR, 'Top 20 words in unsafe scenarios')
]:
    words, counts = zip(*top_words)
    y = np.arange(len(words))
    ax.barh(y, counts, color=color, edgecolor='white', alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(words, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel('Frequency')
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 1.9 Most discriminative words (unsafe vs safe ratio)

Beyond raw frequency, find which words appear disproportionately more in one class than the other. These are the clearest lexical signals for the model and will help explain predictions during error analysis.

In [ ]:
safe_freq   = Counter(re.findall(r'\b[a-z]{4,}\b', ' '.join(safe_texts).lower()))
unsafe_freq = Counter(re.findall(r'\b[a-z]{4,}\b', ' '.join(unsafe_texts).lower()))
all_vocab   = set(safe_freq) | set(unsafe_freq)

rows_ratio = []
for w in all_vocab:
    if w in STOPWORDS:
        continue
    s = safe_freq.get(w, 0)
    u = unsafe_freq.get(w, 0)
    if s + u < 15:
        continue
    rows_ratio.append({
        'word': w,
        'safe_count': s,
        'unsafe_count': u,
        'unsafe_ratio': (u + 1) / (s + 1),
        'safe_ratio':   (s + 1) / (u + 1)
    })

ratio_df = pd.DataFrame(rows_ratio)

top_unsafe = ratio_df.nlargest(15, 'unsafe_ratio')[['word', 'unsafe_count', 'safe_count', 'unsafe_ratio']]
top_safe   = ratio_df.nlargest(15, 'safe_ratio')[['word', 'safe_count', 'unsafe_count', 'safe_ratio']]

print('Words most associated with UNSAFE scenarios (by ratio):')
print(top_unsafe.to_string(index=False))

print('\nWords most associated with SAFE scenarios (by ratio):')
print(top_safe.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, color, col, title in [
    (axes[0], top_unsafe, UNSAFE_COLOR, 'unsafe_ratio', 'Most discriminative UNSAFE words'),
    (axes[1], top_safe,   SAFE_COLOR,   'safe_ratio',   'Most discriminative SAFE words')
]:
    data = data.sort_values(col)
    ax.barh(data['word'], data[col], color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Frequency ratio (smoothed)')
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

### 1.10 EDA summary

Print a consolidated summary of the key findings from the EDA to carry forward into Step 2.

In [ ]:
print('=' * 55)
print('EDA SUMMARY')
print('=' * 55)
print(f'Total samples       : {len(df):,}')
print(f'Safe (label 0)      : {(df["label"]==0).sum():,}  ({(df["label"]==0).mean()*100:.1f}%)')
print(f'Unsafe (label 1)    : {(df["label"]==1).sum():,}  ({(df["label"]==1).mean()*100:.1f}%)')
print(f'Unique movies       : {df["movie"].nunique()}')
print(f'Avg words/scenario  : {df["word_len"].mean():.1f}')
print(f'Max words/scenario  : {df["word_len"].max()}')
print(f'Over 350 words      : {(df["word_len"] > 350).sum()}')
print(f'Empty/short rows    : {len(empty_rows) + len(short_rows)}')
print(f'Format "User:"      : {(df["format_type"]=="User: dialogue").sum()}')
print(f'Format "Scenario N": {(df["format_type"]=="Scenario N: narrative").sum()}')
print(f'Format other        : {(df["format_type"]=="Other / free text").sum()}')
print('=' * 55)
print('Key takeaways:')
print('  Classes are nearly balanced — no resampling needed.')
print('  All samples fit within 512 tokens — BERT is safe to use.')
print('  3 malformed rows should be removed before training.')
print('  Strong lexical signals exist (worthless, scum, filthy...).')
print('  LLM-generated labels may contain noise — plan for Step 4.')

---
# Step 2: Preprocessing and Feature Engineering

### 2.1 Remove malformed rows

Drop empty and near-empty scenarios identified in Step 1. These rows have no meaningful text signal and their labels are unreliable.

In [ ]:
original_len = len(df)

df_clean = df[df['scenario'].fillna('').str.strip().str.len() >= 20].copy()
df_clean = df_clean.reset_index(drop=True)

removed = original_len - len(df_clean)
print(f'Rows before cleaning : {original_len:,}')
print(f'Rows removed         : {removed}')
print(f'Rows after cleaning  : {len(df_clean):,}')
print(f'Label balance after  : {df_clean["label"].value_counts(normalize=True).round(3).to_dict()}')

### 2.2 Text normalisation

Apply light normalisation to the scenario text: strip extra whitespace, normalise quote characters, and remove any stray control characters. Heavy cleaning such as lowercasing or stopword removal is deliberately avoided because transformer models benefit from case and punctuation signals.

In [ ]:
def normalise_text(text):
    text = str(text)
    # Normalise smart quotes to straight quotes
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    # Collapse multiple spaces and strip
    text = re.sub(r'[ \t]+', ' ', text).strip()
    # Remove non-printable control characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text

df_clean['scenario_clean'] = df_clean['scenario'].apply(normalise_text)
df_clean['movie_clean']    = df_clean['movie'].apply(normalise_text)

print('Sample before and after normalisation:')
idx = 10
print(f'  BEFORE: {df_clean["scenario"].iloc[idx][:120]}')
print(f'  AFTER:  {df_clean["scenario_clean"].iloc[idx][:120]}')

### 2.3 Combine movie context and scenario into a single input string

Transformer models take a single text input. Prepending the movie name gives the model the character context it needs to interpret the scenario correctly — the same words mean different things depending on the fictional universe. This is the most important feature engineering decision in the whole pipeline.

In [ ]:
def build_input_text(row):
    return f'[Movie: {row["movie_clean"]}] {row["scenario_clean"]}'

df_clean['input_text'] = df_clean.apply(build_input_text, axis=1)

print('Sample combined inputs:')
for i in [0, 1, 5]:
    row = df_clean.iloc[i]
    print(f'  Label={row["label"]}  {row["input_text"][:130]}')
    print()

# Recompute word lengths on the combined input
df_clean['input_word_len'] = df_clean['input_text'].str.split().str.len()
print(f'Combined input word length — mean: {df_clean["input_word_len"].mean():.1f}, '
      f'max: {df_clean["input_word_len"].max()}')
print(f'Samples over 350 words: {(df_clean["input_word_len"] > 350).sum()}')

### 2.4 Train / validation split

Split the cleaned data into 80% training and 20% validation sets. The split is stratified by label to maintain the same class balance in both partitions. A fixed random seed ensures reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

X = df_clean['input_text'].values
y = df_clean['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f'Training set   : {len(X_train):,} samples')
print(f'Validation set : {len(X_val):,} samples')
print(f'Train label balance : safe={np.mean(y_train==0)*100:.1f}%  unsafe={np.mean(y_train==1)*100:.1f}%')
print(f'Val   label balance : safe={np.mean(y_val==0)*100:.1f}%  unsafe={np.mean(y_val==1)*100:.1f}%')

### 2.5 TF-IDF feature extraction (baseline features)

Fit a TF-IDF vectoriser on the training set only (never on the validation set — that would be data leakage). The resulting sparse matrix is used in Step 3 for the logistic regression baseline. Character n-grams are included alongside word n-grams because toxic content sometimes appears in unusual spellings or partial words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Word n-gram TF-IDF
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    min_df=2,
    sublinear_tf=True
)

# Character n-gram TF-IDF
char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=30_000,
    min_df=3,
    sublinear_tf=True
)

X_train_word = word_tfidf.fit_transform(X_train)
X_val_word   = word_tfidf.transform(X_val)

X_train_char = char_tfidf.fit_transform(X_train)
X_val_char   = char_tfidf.transform(X_val)

# Combine word and char features
X_train_tfidf = hstack([X_train_word, X_train_char])
X_val_tfidf   = hstack([X_val_word,   X_val_char])

print(f'TF-IDF training matrix shape : {X_train_tfidf.shape}')
print(f'TF-IDF validation matrix shape: {X_val_tfidf.shape}')
print(f'Word vocab size  : {len(word_tfidf.vocabulary_):,}')
print(f'Char vocab size  : {len(char_tfidf.vocabulary_):,}')

### 2.6 Hand-crafted features

Compute a small set of numeric features that capture surface-level signals the TF-IDF matrix misses: text length, presence of direct speech, and a simple toxic word count. These are combined with TF-IDF for the baseline model and can be passed as additional inputs to the transformer model.

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix

TOXIC_WORDS = {
    'worthless', 'scum', 'filthy', 'fools', 'stupid', 'unworthy',
    'idiot', 'moron', 'trash', 'loser', 'pathetic', 'disgusting',
    'coward', 'weak', 'useless', 'inferior', 'vermin'
}

def extract_hand_features(texts):
    rows = []
    for text in texts:
        words_lower = re.findall(r'\b[a-z]+\b', text.lower())
        rows.append([
            len(text),                                         # char length
            len(words_lower),                                  # word count
            int(text.strip().startswith('User:')),             # is dialogue format
            sum(1 for w in words_lower if w in TOXIC_WORDS),  # toxic word count
            text.count('!'),                                   # exclamation marks
            text.count('?'),                                   # question marks
        ])
    return np.array(rows, dtype=np.float32)

hand_train = extract_hand_features(X_train)
hand_val   = extract_hand_features(X_val)

scaler = StandardScaler()
hand_train_scaled = scaler.fit_transform(hand_train)
hand_val_scaled   = scaler.transform(hand_val)

# Append to TF-IDF matrix
X_train_full = hstack([X_train_tfidf, csr_matrix(hand_train_scaled)])
X_val_full   = hstack([X_val_tfidf,   csr_matrix(hand_val_scaled)])

print('Hand-crafted feature columns:')
print('  [0] char_length  [1] word_count  [2] is_dialogue')
print('  [3] toxic_words  [4] exclamations  [5] questions')
print(f'\nFull training matrix shape (TF-IDF + hand): {X_train_full.shape}')

# Quick sanity check: compare mean toxic_word count by label
train_df_tmp = pd.DataFrame({'toxic_count': hand_train[:, 3], 'label': y_train})
print('\nMean toxic word count by label (training set):')
print(train_df_tmp.groupby('label')['toxic_count'].mean().round(3))

### 2.7 Save preprocessed data

Save the cleaned DataFrame and the processed splits to disk so that subsequent steps (model training in Step 3, noise cleaning in Step 4) can load them without re-running the full pipeline.

In [ ]:
import joblib

# Save cleaned dataframe
df_clean.to_csv('train_clean.csv', index=False)
print('Saved: train_clean.csv')

# Save train/val splits as numpy arrays
np.save('X_train.npy', X_train)
np.save('X_val.npy',   X_val)
np.save('y_train.npy', y_train)
np.save('y_val.npy',   y_val)
print('Saved: X_train.npy, X_val.npy, y_train.npy, y_val.npy')

# Save TF-IDF vectorisers and scaler for later inference
joblib.dump(word_tfidf,       'word_tfidf.pkl')
joblib.dump(char_tfidf,       'char_tfidf.pkl')
joblib.dump(scaler,           'hand_feature_scaler.pkl')
print('Saved: word_tfidf.pkl, char_tfidf.pkl, hand_feature_scaler.pkl')

print('\nStep 2 complete. Ready for Step 3: model training.')

### 2.8 Preprocessing summary

Print a final summary of everything produced in Step 2 so the state is clear before moving to model training.

In [ ]:
print('=' * 55)
print('PREPROCESSING SUMMARY')
print('=' * 55)
print(f'Malformed rows removed       : {original_len - len(df_clean)}')
print(f'Clean samples total          : {len(df_clean):,}')
print(f'Training set size            : {len(X_train):,}')
print(f'Validation set size          : {len(X_val):,}')
print(f'Input format                 : [Movie: <name>] <scenario>')
print(f'TF-IDF feature dimensions    : {X_train_tfidf.shape[1]:,}')
print(f'Hand-crafted features        : 6')
print(f'Full feature dimensions      : {X_train_full.shape[1]:,}')
print(f'Artefacts saved              : 7 files')
print('=' * 55)
print('Next step: Step 3 — baseline (LR) and transformer (BERT) models.')